In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#HEART DISEASE dateset exploratory analysis

df=pd.read_csv('/kaggle/input/heart-disease-dataset/heart.csv')

In [ ]:
df.info()

heart rate parameters

1.Age
2.sex male-1,female-0
3.cp chest pain(0=typical angina,1=atypical angina,2=non anginal pain,3=asymptomatic)
4.tresbps - resting blood pressure
5.chol - cholestol
6.fasting blood sugar indicates diabetes risk 0 false 1 true
7.restecg = normal 1 abonomrality 2 ventricular hypertrophy
8.maxm heart rate caheived 
9exang angina when exercising
10.oldpeak
11.slope
12.ca more blocked blood vessels fluoroscopy
13. thal blood flow abnormalities Thalasemia
14 target - yes ir no heart diease 



In [ ]:
df.head()

In [ ]:
#visulaisation libarires needed

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum

## there are no null values in the dataset

In [ ]:
ax=sns.countplot(x='target',data=df)

## we have almost equal amount of heartdisease and non heart disease patients
so our data set is not biased

In [ ]:
df.hist(figsize=(14,10))
plt.show()

# Age distribution
The age is little skewed towards more than 40
the cholestol is skewed towards left 200mg
the bloodpressure 0 has more count

In [ ]:
sns.boxplot(x='target',y='age',data=df)
sns.boxplot(x='target',y='thalach',data=df)
sns.boxplot(x='target',y='chol',data=df)

## Observation
we can observe here that heart disease(distribution is around 45 to 60)//but 50 to 60 age have no heart disease

## Observation
we can observe here that heart disease(distribution of cholestol is slightly lower for people who have disease

## Observation
we can observe here that heart disease blood flow abonrmalities are higher for heart disease patients

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(),annot=True,cmap='coolwarm')
plt.title("Feature Correlation heatmap")
plt.show()

## Observations:
\1.For target possitive corelations are CP,THALACH,SLOPE\

2.it also has strong negative corelation with EXANG,oldpeak

In [ ]:
corr_target=df.corr()['target'].sort_values(ascending=False)
corr_target

In [ ]:
sns.boxplot(x='target',y='cp',data=df)

In [ ]:
sns.boxplot(x='target',y='thalach',data=df)

In [ ]:
sns.boxplot(x='target',y='slope',data=df)

In [ ]:
X=df.drop('target',axis=1)
y=df['target']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)


In [ ]:
y_pred=model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)
print("Accuracy:", rf_accuracy)

print("\nClassification Report:\n", classification_report(y_test, rf_pred))

In [ ]:
print("Logistic Regression Accuracy:", accuracy)
print("Random Forest Accuracy:", rf_accuracy)

# The accuracy of random forest model is better than logistic regression

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

importance.head(10)

In [ ]:
importance.head(10).plot(kind='bar')

### Feature Importance Insights

The Random Forest model identifies chest pain type (cp), number of major vessels (ca), thalassemia test results (thal), and ST depression (oldpeak) as the most influential predictors. These variables are clinically associated with coronary artery abnormalities and exercise-induced cardiac stress, making them strong indicators for heart disease prediction.

## ROC-AUC Evaluation
Evaluating model discrimination ability using ROC-AUC score.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

rf_probs = rf_model.predict_proba(X_test)[:,1]
roc_score = roc_auc_score(y_test, rf_probs)

print("ROC-AUC Score:", roc_score)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, rf_probs)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

The ROC curve demonstrates the model’s ability to distinguish between patients with and without heart disease across different classification thresholds.

## Hyperparameter Tuning
Improving Random Forest performance using parameter optimization.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':[100,200],
    'max_depth':[4,6,8,None]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid,
                    cv=5)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

In [ ]:
best_pred = best_model.predict(X_test)
best_acc = accuracy_score(y_test, best_pred)

print("Tuned Model Accuracy:", best_acc)